In [21]:
%pip install pandas numpy sklearn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SK

In [22]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

from sklearn.model_selection import train_test_split


In [23]:


# ------------------------------------------
# Load Dataset
# ------------------------------------------

INPUT_PATH = "../data/processed/cleaned_data.csv"

df = pd.read_csv(INPUT_PATH)

print("Dataset Loaded Successfully")

print("\nShape:")
print(df.shape)


Dataset Loaded Successfully

Shape:
(10000, 18)


In [24]:

# ==========================================
# REMOVE IRRELEVANT COLUMNS
# ==========================================

columns_to_drop = [
    "patient_id",
    "name",
    "notes"
]

existing_columns = [
    col for col in columns_to_drop
    if col in df.columns
]

df.drop(columns=existing_columns, inplace=True)

print("\nRemoved unnecessary columns")



Removed unnecessary columns


In [25]:

# ==========================================
# FEATURE ENGINEERING
# ==========================================

print("\n===================================")
print("FEATURE ENGINEERING")
print("===================================")



FEATURE ENGINEERING


In [26]:

# ------------------------------------------
# 1. Pulse Pressure
# ------------------------------------------

if (
    "systolic_bp" in df.columns and
    "diastolic_bp" in df.columns
):

    df["pulse_pressure"] = (
        df["systolic_bp"] -
        df["diastolic_bp"]
    )


In [27]:

# ------------------------------------------
# 2. Mean Arterial Pressure
# ------------------------------------------

if (
    "systolic_bp" in df.columns and
    "diastolic_bp" in df.columns
):

    df["mean_arterial_pressure"] = (
        df["diastolic_bp"] +
        (
            (
                df["systolic_bp"] -
                df["diastolic_bp"]
            ) / 3
        )
    )


In [28]:

# ------------------------------------------
# 3. BMI Category
# ------------------------------------------

def bmi_category(bmi):

    if bmi < 18.5:
        return "underweight"

    elif bmi < 25:
        return "normal"

    elif bmi < 30:
        return "overweight"

    else:
        return "obese"

if "bmi" in df.columns:

    df["bmi_category"] = (
        df["bmi"]
        .apply(bmi_category)
    )


In [29]:

# ------------------------------------------
# 4. Age Group
# ------------------------------------------

if "age" in df.columns:

    bins = [0, 18, 35, 50, 65, 120]

    labels = [
        "child",
        "young_adult",
        "adult",
        "middle_age",
        "senior"
    ]

    df["age_group"] = pd.cut(
        df["age"],
        bins=bins,
        labels=labels
    )


In [30]:

# ------------------------------------------
# 5. High Risk Flag
# ------------------------------------------

df["high_risk"] = 0

risk_condition = (
    (
        df["cholesterol_level"] > 240
    ) |
    (
        df["bmi"] > 30
    ) |
    (
        df["heart_rate"] > 100
    ) |
    (
        df["systolic_bp"] > 140
    )
)

df.loc[risk_condition, "high_risk"] = 1

# ------------------------------------------
# 6. Follow-up urgency
# ------------------------------------------

if "follow_up" in df.columns:

    df["urgent_followup"] = np.where(
        df["follow_up"] <= 7,
        1,
        0
    )


In [31]:

# ==========================================
# DATE FEATURE EXTRACTION
# ==========================================

print("\n===================================")
print("DATE FEATURE EXTRACTION")
print("===================================")

if "last_visit_date" in df.columns:

    df["last_visit_date"] = pd.to_datetime(
        df["last_visit_date"],
        errors="coerce"
    )

    df["visit_year"] = (
        df["last_visit_date"]
        .dt.year
    )

    df["visit_month"] = (
        df["last_visit_date"]
        .dt.month
    )

    df["visit_day"] = (
        df["last_visit_date"]
        .dt.day
    )

# Drop original date column
if "last_visit_date" in df.columns:

    df.drop(
        columns=["last_visit_date"],
        inplace=True
    )



DATE FEATURE EXTRACTION


In [32]:

# ==========================================
# HANDLE MISSING VALUES
# ==========================================

print("\n===================================")
print("MISSING VALUE HANDLING")
print("===================================")

# Numerical columns
numeric_cols = df.select_dtypes(
    include=np.number
).columns

for col in numeric_cols:

    df[col] = df[col].fillna(
        df[col].median()
    )

# Categorical columns
categorical_cols = df.select_dtypes(
    include="object"
).columns

for col in categorical_cols:

    df[col] = df[col].fillna(
        df[col].mode()[0]
    )



MISSING VALUE HANDLING


/var/folders/fv/8r8v80lx31l_v__pps6svvpr0000gn/T/ipykernel_91725/1290748289.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


In [33]:

# ==========================================
# ENCODING CATEGORICAL VARIABLES
# ==========================================

print("\n===================================")
print("ENCODING")
print("===================================")

label_encoders = {}

categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_columns:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    label_encoders[col] = le

    print(f"Encoded: {col}")



ENCODING
Encoded: gender
Encoded: city
Encoded: smoker
Encoded: medications
Encoded: diagnosis_code
Encoded: bmi_category
Encoded: age_group


/var/folders/fv/8r8v80lx31l_v__pps6svvpr0000gn/T/ipykernel_91725/1427169743.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


In [34]:

# ==========================================
# FEATURE SCALING
# ==========================================

print("\n===================================")
print("FEATURE SCALING")
print("===================================")

target_column = "has_disease"

feature_columns = [
    col for col in df.columns
    if col != target_column
]

X = df[feature_columns]

y = df[target_column]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(
    X_scaled,
    columns=feature_columns
)



FEATURE SCALING


In [35]:

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

print("\n===================================")
print("TRAIN TEST SPLIT")
print("===================================")

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)

print("Testing Shape:", X_test.shape)



TRAIN TEST SPLIT
Training Shape: (8000, 22)
Testing Shape: (2000, 22)


In [36]:

# ==========================================
# SAVE FINAL DATASETS
# ==========================================

print("\n===================================")
print("SAVING FILES")
print("===================================")

# Final complete dataset
final_dataset = pd.concat(
    [X_scaled_df, y.reset_index(drop=True)],
    axis=1
)

final_dataset.to_csv(
    "../data/processed/final_dataset.csv",
    index=False
)

# Train/Test datasets

X_train.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Files saved successfully")



SAVING FILES
Files saved successfully


In [37]:

# ==========================================
# FINAL OUTPUT SUMMARY
# ==========================================

print("\n===================================")
print("FINAL DATASET SUMMARY")
print("===================================")

print("\nFinal Dataset Shape:")
print(final_dataset.shape)

print("\nFinal Columns:")
print(final_dataset.columns.tolist())

print("\nTarget Distribution:")
print(
    final_dataset["has_disease"]
    .value_counts()
)

print("\nPreview:")
print(final_dataset.head())

print("\nML-ready dataset prepared successfully")


FINAL DATASET SUMMARY

Final Dataset Shape:
(10000, 23)

Final Columns:
['age', 'gender', 'city', 'bmi', 'heart_rate', 'cholesterol_level', 'diabetic', 'smoker', 'medications', 'follow_up', 'diagnosis_code', 'systolic_bp', 'diastolic_bp', 'pulse_pressure', 'mean_arterial_pressure', 'bmi_category', 'age_group', 'high_risk', 'urgent_followup', 'visit_year', 'visit_month', 'visit_day', 'has_disease']

Target Distribution:
has_disease
0.0    7539
1.0    2461
Name: count, dtype: int64

Preview:
        age    gender      city       bmi  heart_rate  cholesterol_level  \
0  0.006183  1.203037 -1.668945 -0.319109    0.151923          -0.675982   
1  0.006183 -1.242657  0.416975  0.910723   -0.128188          -1.022533   
2  0.006183  1.203037  0.416975 -0.319109   -0.128188          -0.675982   
3  2.046922 -1.242657  0.416975  3.370386   -0.128188           1.403324   
4  0.006183 -0.019810 -0.278331  2.302374   -0.128188          -1.022533   

   diabetic    smoker  medications  follow_up  